# TCN Solar Forecaster

## 1. Train/Validation/Test Split

In [4]:
"""
Train/Val/Test split + Min-Max normalization for TCN.

Split (as agreed):
  Train: 2019 + 2020 RL-train days (282) + 2020 other
  Val:   early 2021
  Test:  73 RL-test days (2020) + late 2021

Normalization: fit on train only, apply to all.
"""

import pandas as pd
import numpy as np

# ============================================================
# 1. Load features and RL date lists
# ============================================================
df = pd.read_csv('baalbeck_features.csv', index_col=0, parse_dates=True)
rl_test_dates = pd.read_csv('rl_test_dates.csv', parse_dates=['date'])['date']

rl_test_set = set(rl_test_dates.dt.normalize())

total_hours = len(df)
print(f"Total: {total_hours} hours")
print(f"Features: {len(df.columns)}")

# ============================================================
# 2. Assign blocks
# ============================================================

# 2020 RL-test (fixed in test)
mask_rl_test = df.index.normalize().isin(rl_test_set)
df_rl_test = df[mask_rl_test]

# Train: everything before 2021, excluding RL-test days
df_pre2021 = df[df.index.year < 2021]
df_train = df_pre2021[~df_pre2021.index.normalize().isin(rl_test_set)]

# 2021 data: split into val (early) and additional test (late)
# Snap to day boundary
df_2021 = df[df.index.year == 2021]
additional_test_needed = int(0.15 * total_hours) - len(df_rl_test)
approx_val_hours = len(df_2021) - additional_test_needed

# Find the last complete day that fits within approx_val_hours
cutoff_timestamp = df_2021.index[approx_val_hours - 1]
cutoff_date = cutoff_timestamp.normalize()  # start of that day

# Val = all days up to and including cutoff_date
# Test = all days after cutoff_date
df_val = df_2021[df_2021.index.normalize() <= cutoff_date]
df_additional_test = df_2021[df_2021.index.normalize() > cutoff_date]

# Combine test
df_test = pd.concat([df_rl_test, df_additional_test]).sort_index()

# ============================================================
# 3. Print split details
# ============================================================
print(f"\n{'='*60}")
print("SPLIT SUMMARY")
print(f"{'='*60}")

print(f"\nTrain: {len(df_train)} hours ({len(df_train)/total_hours*100:.1f}%)")
print(f"  Range: {df_train.index[0]} to {df_train.index[-1]}")
print(f"  2019: {len(df_train[df_train.index.year==2019])} hours")
print(f"  2020: {len(df_train[df_train.index.year==2020])} hours")

print(f"\nVal:   {len(df_val)} hours ({len(df_val)/total_hours*100:.1f}%)")
print(f"  Range: {df_val.index[0]} to {df_val.index[-1]}")

print(f"\nTest:  {len(df_test)} hours ({len(df_test)/total_hours*100:.1f}%)")
print(f"  = 73 RL-test days ({len(df_rl_test)}) + 2021 late ({len(df_additional_test)})")
if len(df_rl_test) > 0:
    print(f"  RL-test range: {df_rl_test.index[0]} to {df_rl_test.index[-1]}")
if len(df_additional_test) > 0:
    print(f"  2021 late range: {df_additional_test.index[0]} to {df_additional_test.index[-1]}")

print(f"\nSum: {len(df_train) + len(df_val) + len(df_test)} (should be {total_hours})")

# ============================================================
# 4. Leakage check
# ============================================================
print(f"\n{'='*60}")
print("LEAKAGE CHECK")
print(f"{'='*60}")

train_dates = set(df_train.index.normalize())
val_dates = set(df_val.index.normalize())
test_dates = set(df_test.index.normalize())

leak_train = train_dates & rl_test_set
leak_val = val_dates & rl_test_set

print(f"  RL-test days in train: {len(leak_train)} (should be 0)")
print(f"  RL-test days in val:   {len(leak_val)} (should be 0)")
print(f"  Train-Val overlap:     {len(train_dates & val_dates)} (should be 0)")
print(f"  Train-Test overlap:    {len(train_dates & test_dates)} (should be 0)")
print(f"  Val-Test overlap:      {len(val_dates & test_dates)} (should be 0)")

# ============================================================
# 5. Min-Max normalization (fit on train only)
# ============================================================
print(f"\n{'='*60}")
print("MIN-MAX NORMALIZATION")
print(f"{'='*60}")

train_min = df_train.min()
train_max = df_train.max()
train_range = train_max - train_min

# Avoid division by zero for constant columns
train_range[train_range == 0] = 1.0

df_train_norm = (df_train - train_min) / train_range
df_val_norm = (df_val - train_min) / train_range
df_test_norm = (df_test - train_min) / train_range

print("\nTrain min/max per feature:")
for col in df.columns:
    print(f"  {col:<20s}: min={train_min[col]:>10.4f}  max={train_max[col]:>10.4f}")

print(f"\nNormalized ranges:")
print(f"  Train: [{df_train_norm.min().min():.4f}, {df_train_norm.max().max():.4f}]")
print(f"  Val:   [{df_val_norm.min().min():.4f}, {df_val_norm.max().max():.4f}]")
print(f"  Test:  [{df_test_norm.min().min():.4f}, {df_test_norm.max().max():.4f}]")

val_outside = (df_val_norm < 0).sum().sum() + (df_val_norm > 1).sum().sum()
test_outside = (df_test_norm < 0).sum().sum() + (df_test_norm > 1).sum().sum()
print(f"\n  Val values outside [0,1]: {val_outside}")
print(f"  Test values outside [0,1]: {test_outside}")

# ============================================================
# 6. Save
# ============================================================
df_train_norm.to_csv('tcn_train.csv')
df_val_norm.to_csv('tcn_val.csv')
df_test_norm.to_csv('tcn_test.csv')

norm_params = pd.DataFrame({'min': train_min, 'max': train_max, 'range': train_range})
norm_params.to_csv('tcn_norm_params.csv')

print(f"\nSaved:")
print(f"  tcn_train.csv       ({len(df_train_norm)} rows)")
print(f"  tcn_val.csv         ({len(df_val_norm)} rows)")
print(f"  tcn_test.csv        ({len(df_test_norm)} rows)")
print(f"  tcn_norm_params.csv (for inverse transform)")

Total: 17714 hours
Features: 17

SPLIT SUMMARY

Train: 12620 hours (71.2%)
  Range: 2019-03-04 13:00:00 to 2020-12-31 21:00:00
  2019: 6806 hours
  2020: 5814 hours

Val:   2446 hours (13.8%)
  Range: 2021-01-01 00:00:00 to 2021-04-12 23:00:00

Test:  2648 hours (14.9%)
  = 73 RL-test days (1472) + 2021 late (1176)
  RL-test range: 2020-01-01 05:00:00 to 2020-12-30 23:00:00
  2021 late range: 2021-04-13 00:00:00 to 2021-05-31 23:00:00

Sum: 17714 (should be 17714)

LEAKAGE CHECK
  RL-test days in train: 0 (should be 0)
  RL-test days in val:   0 (should be 0)
  Train-Val overlap:     0 (should be 0)
  Train-Test overlap:    0 (should be 0)
  Val-Test overlap:      0 (should be 0)

MIN-MAX NORMALIZATION

Train min/max per feature:
  GHI                 : min=    0.0000  max= 1094.7640
  DNI                 : min=    0.0000  max= 1790.1722
  DHI                 : min=    0.0000  max=  850.2751
  Temperature         : min=   -4.2748  max=   39.4405
  Humidity            : min=    8.0880  

## 2. Sliding Window Samples

In [5]:
"""
Create sliding window samples for TCN training.

For each sample:
  Input:  24 consecutive hours × 17 features (shape: 24 × 17)
  Target: next 24 consecutive hours of kt only (shape: 24)

Skips any sample where the 48h window spans a time gap
(from sensor outages or removed RL-test days).
"""

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1. Load normalized data
# ============================================================
df_train = pd.read_csv('tcn_train.csv', index_col=0, parse_dates=True)
df_val = pd.read_csv('tcn_val.csv', index_col=0, parse_dates=True)
df_test = pd.read_csv('tcn_test.csv', index_col=0, parse_dates=True)

print(f"Train hours: {len(df_train)}")
print(f"Val hours:   {len(df_val)}")
print(f"Test hours:  {len(df_test)}")

# ============================================================
# 2. Settings
# ============================================================
INPUT_HOURS = 24
TARGET_HOURS = 24
TOTAL_WINDOW = INPUT_HOURS + TARGET_HOURS
TARGET_COL = 'kt'

all_columns = list(df_train.columns)
kt_index = all_columns.index(TARGET_COL)
print(f"\nTarget column: '{TARGET_COL}' (index {kt_index})")
print(f"Input window: {INPUT_HOURS}h, Target window: {TARGET_HOURS}h")

# ============================================================
# 3. Create samples with continuity check only
# ============================================================
def create_samples(df):
    data = df.values
    dates = df.index
    
    X_list = []
    y_list = []
    skipped = 0
    
    for i in range(len(df) - TOTAL_WINDOW + 1):
        time_diff = dates[i + TOTAL_WINDOW - 1] - dates[i]
        expected_diff = pd.Timedelta(hours=TOTAL_WINDOW - 1)
        if time_diff != expected_diff:
            skipped += 1
            continue
        
        X = data[i : i + INPUT_HOURS, :]
        y = data[i + INPUT_HOURS : i + TOTAL_WINDOW, kt_index]
        
        X_list.append(X)
        y_list.append(y)
    
    return np.array(X_list), np.array(y_list), skipped

# ============================================================
# 4. Create samples for each set
# ============================================================
print(f"\n{'='*60}")
print("CREATING SAMPLES")
print(f"{'='*60}")

print(f"\nTraining ({len(df_train)} hours)...")
X_train, y_train, skipped_train = create_samples(df_train)
print(f"  Samples: {len(X_train)}, Skipped: {skipped_train}")

print(f"\nValidation ({len(df_val)} hours)...")
X_val, y_val, skipped_val = create_samples(df_val)
print(f"  Samples: {len(X_val)}, Skipped: {skipped_val}")

print(f"\nTest ({len(df_test)} hours)...")
X_test, y_test, skipped_test = create_samples(df_test)
print(f"  Samples: {len(X_test)}, Skipped: {skipped_test}")

# ============================================================
# 5. Summary
# ============================================================
print(f"\n{'='*60}")
print("FINAL DATASET SUMMARY")
print(f"{'='*60}")
print(f"  Train:  {X_train.shape[0]} samples, X: {X_train.shape}, y: {y_train.shape}")
print(f"  Val:    {X_val.shape[0]} samples, X: {X_val.shape}, y: {y_val.shape}")
print(f"  Test:   {X_test.shape[0]} samples, X: {X_test.shape}, y: {y_test.shape}")

# ============================================================
# 6. PyTorch datasets and dataloaders
# ============================================================
class TCNDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X).permute(0, 2, 1)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 32

train_dataset = TCNDataset(X_train, y_train)
val_dataset = TCNDataset(X_val, y_val)
test_dataset = TCNDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n{'='*60}")
print("PYTORCH DATALOADERS READY")
print(f"{'='*60}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Train batches/epoch: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

X_batch, y_batch = next(iter(train_loader))
print(f"\n  Sample batch:")
print(f"    X: {X_batch.shape}  (batch, features, hours)")
print(f"    y: {y_batch.shape}  (batch, hours)")

# ============================================================
# 7. Save numpy arrays
# ============================================================
np.save('X_train.npy', X_train)
np.save('y_train.npy', y_train)
np.save('X_val.npy', X_val)
np.save('y_val.npy', y_val)
np.save('X_test.npy', X_test)
np.save('y_test.npy', y_test)

print(f"\nSaved numpy arrays for reuse.")

Train hours: 12620
Val hours:   2446
Test hours:  2648

Target column: 'kt' (index 12)
Input window: 24h, Target window: 24h

CREATING SAMPLES

Training (12620 hours)...
  Samples: 9276, Skipped: 3297

Validation (2446 hours)...
  Samples: 2352, Skipped: 47

Test (2648 hours)...
  Samples: 1405, Skipped: 1196

FINAL DATASET SUMMARY
  Train:  9276 samples, X: (9276, 24, 17), y: (9276, 24)
  Val:    2352 samples, X: (2352, 24, 17), y: (2352, 24)
  Test:   1405 samples, X: (1405, 24, 17), y: (1405, 24)

PYTORCH DATALOADERS READY
  Batch size: 32
  Train batches/epoch: 290
  Val batches: 74
  Test batches: 44

  Sample batch:
    X: torch.Size([32, 17, 24])  (batch, features, hours)
    y: torch.Size([32, 24])  (batch, hours)

Saved numpy arrays for reuse.


## 3. TCN Architecture and Model Loading

In [3]:
"""
Pre-compute TCN predictions for all hours in 2020.
Uses Trial 2 model (17 features, 32 filters, LR=0.0001).
These will be looked up by the RL environment during training.
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ============================================================
# 1. TCN Architecture (copied from training script)
# ============================================================
class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size,
                               dilation=dilation, padding=self.padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size,
                               dilation=dilation, padding=self.padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        if in_channels != out_channels:
            self.residual = nn.Conv1d(in_channels, out_channels, 1)
        else:
            self.residual = None

        self.relu_out = nn.ReLU()

    def forward(self, x):
        out = self.conv1(x)
        out = out[:, :, :x.size(2)]
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = out[:, :, :x.size(2)]
        out = self.relu2(out)
        out = self.dropout2(out)

        if self.residual is not None:
            x = self.residual(x)

        return self.relu_out(out + x)

class TCNForecaster(nn.Module):
    def __init__(self, num_features, num_filters, kernel_size, dilations, dropout):
        super().__init__()
        blocks = []
        in_channels = num_features
        for d in dilations:
            blocks.append(TCNBlock(in_channels, num_filters, kernel_size, d, dropout))
            in_channels = num_filters
        self.tcn = nn.Sequential(*blocks)
        self.output_conv = nn.Conv1d(num_filters, 1, 1)

    def forward(self, x):
        out = self.tcn(x)
        out = self.output_conv(out)
        out = out.squeeze(1)
        return out

# ============================================================
# 2. Load Trial 2 model (same config as training script)
# ============================================================
TRIAL_DIR = 'TCN_Model'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = TCNForecaster(
    num_features=17,
    num_filters=32,
    kernel_size=3,
    dilations=[1, 2, 4, 8],
    dropout=0.3
).to(device)

model.load_state_dict(torch.load(f'{TRIAL_DIR}/model.pt', map_location=device))
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"TCN model loaded from {TRIAL_DIR}")
print(f"Parameters: {total_params:,}")
print(f"Device: {device}")



TCN model loaded from TCN_Model
Parameters: 24,001
Device: cuda


## 4. Test Set Performance

In [7]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# ============================================================
# Load test data and normalization params
# ============================================================
X_test = np.load('X_test.npy')
y_test = np.load('y_test.npy')
norm_params = pd.read_csv('tcn_norm_params.csv', index_col=0)

INPUT_HOURS = 24
TARGET_HOURS = 24
TOTAL_WINDOW = INPUT_HOURS + TARGET_HOURS

# ============================================================
# Get GHI_clearsky for test target hours (for kt -> GHI)
# ============================================================
df_test_raw = pd.read_csv('tcn_test.csv', index_col=0, parse_dates=True)

test_clearsky_targets = []
for i in range(len(df_test_raw) - TOTAL_WINDOW + 1):
    time_diff = df_test_raw.index[i + TOTAL_WINDOW - 1] - df_test_raw.index[i]
    expected_diff = pd.Timedelta(hours=TOTAL_WINDOW - 1)
    if time_diff != expected_diff:
        continue
    target_clearsky = df_test_raw['GHI_clearsky'].iloc[i + INPUT_HOURS : i + TOTAL_WINDOW].values
    test_clearsky_targets.append(target_clearsky)

test_clearsky_targets = np.array(test_clearsky_targets)

# ============================================================
# Run trained model on test set
# ============================================================
class TCNDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X).permute(0, 2, 1)
        self.y = torch.FloatTensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

test_loader = DataLoader(TCNDataset(X_test, y_test), batch_size=32, shuffle=False)

all_predictions = []
all_targets = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        predictions = model(X_batch)
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(y_batch.numpy())

all_predictions = np.concatenate(all_predictions, axis=0)
all_targets = np.concatenate(all_targets, axis=0)

# ============================================================
# Denormalize and convert to GHI
# ============================================================
kt_min = norm_params.loc['kt', 'min']
kt_range = norm_params.loc['kt', 'range']
cs_min = norm_params.loc['GHI_clearsky', 'min']
cs_range = norm_params.loc['GHI_clearsky', 'range']

pred_kt = all_predictions * kt_range + kt_min
target_kt = all_targets * kt_range + kt_min
clearsky_denorm = test_clearsky_targets * cs_range + cs_min

pred_ghi = pred_kt * clearsky_denorm
target_ghi = target_kt * clearsky_denorm

# ============================================================
# Persistence baseline (last 24h input kt carried forward)
# ============================================================
kt_index_input = list(pd.read_csv('tcn_test.csv', index_col=0, nrows=0).columns).index('kt')
persist_kt = X_test[:, :, kt_index_input] * kt_range + kt_min
persist_ghi = persist_kt * clearsky_denorm

# ============================================================
# Overall metrics (GHI W/m²)
# ============================================================
model_mae_total = np.mean(np.abs(pred_ghi - target_ghi))
model_rmse_total = np.sqrt(np.mean((pred_ghi - target_ghi) ** 2))
model_mbe_total = np.mean(pred_ghi - target_ghi)
persist_mae_total = np.mean(np.abs(persist_ghi - target_ghi))

print(f"{'='*60}")
print("TCN TEST-SET PERFORMANCE (GHI W/m²)")
print(f"{'='*60}")
print(f"  Model MAE:       {model_mae_total:.2f} W/m²")
print(f"  Model RMSE:      {model_rmse_total:.2f} W/m²")
print(f"  Model MBE:       {model_mbe_total:+.2f} W/m²")
print(f"  Persistence MAE: {persist_mae_total:.2f} W/m²")
print(f"  Skill Score:     {1 - model_mae_total/persist_mae_total:+.2%}")

TCN TEST-SET PERFORMANCE (GHI W/m²)
  Model MAE:       21.13 W/m²
  Model RMSE:      52.72 W/m²
  Model MBE:       +1.24 W/m²
  Persistence MAE: 27.38 W/m²
  Skill Score:     +22.85%


## 5. Pre-compute kt Forecasts for RL

In [8]:

# ============================================================
# 3. Load quality-controlled features for 2020
#    (NOT the imputed RL data — TCN uses clean data only)
# ============================================================
df_features = pd.read_csv('baalbeck_features.csv', index_col=0, parse_dates=True)
df_2020 = df_features[df_features.index.year == 2020].copy()

# Remove Feb 29
df_2020 = df_2020[~((df_2020.index.month == 2) & (df_2020.index.day == 29))]

print(f"\nQuality-controlled 2020 hours: {len(df_2020)}")

# Load normalization params (from TCN training)
norm_params = pd.read_csv('tcn_norm_params.csv', index_col=0)

# Normalize using TCN training stats
feature_cols = [c for c in norm_params.index if c in df_2020.columns]
df_norm = (df_2020[feature_cols] - norm_params.loc[feature_cols, 'min']) / norm_params.loc[feature_cols, 'range']
print(f"Normalized features: {df_norm.shape}")

# ============================================================
# 4. Create complete 2020 hourly index (excl Feb 29)
# ============================================================
full_index = pd.date_range('2020-01-01 00:00', '2020-12-31 23:00', freq='h')
full_index = full_index[~((full_index.month == 2) & (full_index.day == 29))]

print(f"Full 2020 index: {len(full_index)} hours")

# ============================================================
# 5. Pre-compute predictions
#    For each hour h, if the previous 24 hours of clean data exist,
#    run the TCN to predict the next 24 hours of kt.
# ============================================================
INPUT_HOURS = 24
kt_min = norm_params.loc['kt', 'min']
kt_range = norm_params.loc['kt', 'range']

# predictions[h] = predicted kt for hour h
predictions_kt = np.full(len(full_index), np.nan)

# Track which hours have clean data
clean_hours = set(df_norm.index)

valid_predictions = 0
fallback_persistence = 0

print(f"\nPre-computing TCN predictions...")

with torch.no_grad():
    for i in range(len(full_index)):
        if i < INPUT_HOURS:
            continue
        
        if not np.isnan(predictions_kt[i]):
            continue
        
        # Get the 24 input hours
        input_hours = full_index[i - INPUT_HOURS : i]
        
        # Check all 24 input hours have clean data
        if not all(h in clean_hours for h in input_hours):
            continue
        
        # Extract and normalize input
        input_data = df_norm.loc[input_hours].values  # (24, 17)
        
        # Run TCN
        x = torch.FloatTensor(input_data).unsqueeze(0).permute(0, 2, 1).to(device)
        pred_kt_norm = model(x).cpu().numpy()[0]  # (24,)
        
        # Denormalize
        pred_kt = pred_kt_norm * kt_range + kt_min
        pred_kt = np.clip(pred_kt, 0, 1.5)
        
        # Store predictions for the next 24 hours
        for j in range(24):
            target_idx = i + j
            if target_idx < len(full_index):
                if np.isnan(predictions_kt[target_idx]):
                    predictions_kt[target_idx] = pred_kt[j]
        
        valid_predictions += 1

# Fill remaining NaN with persistence
for i in range(len(full_index)):
    if np.isnan(predictions_kt[i]):
        h = full_index[i]
        if h in clean_hours:
            kt_col_idx = feature_cols.index('kt')
            predictions_kt[i] = df_2020.loc[h, 'kt']
        elif i > 0 and not np.isnan(predictions_kt[i - 1]):
            predictions_kt[i] = predictions_kt[i - 1]
        else:
            predictions_kt[i] = 0.0
        fallback_persistence += 1

print(f"\n  Valid TCN predictions: {valid_predictions} windows")
print(f"  Persistence fallback: {fallback_persistence} hours")
print(f"  Coverage: {(1 - fallback_persistence / len(full_index)) * 100:.1f}%")
print(f"  Predicted kt range: {predictions_kt.min():.4f} to {predictions_kt.max():.4f}")
print(f"  NaN remaining: {np.isnan(predictions_kt).sum()}")

# ============================================================
# 6. Save
# ============================================================
np.save('tcn_kt_predictions_2020.npy', predictions_kt)
print(f"\nSaved: tcn_predictions_2020.npy ({len(predictions_kt)} hours)")


Quality-controlled 2020 hours: 7262
Normalized features: (7262, 17)
Full 2020 index: 8760 hours

Pre-computing TCN predictions...


C:\Users\afifb\AppData\Local\Temp\ipykernel_19172\2772362503.py:68: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  x = torch.FloatTensor(input_data).unsqueeze(0).permute(0, 2, 1).to(device)



  Valid TCN predictions: 220 windows
  Persistence fallback: 3487 hours
  Coverage: 60.2%
  Predicted kt range: 0.0000 to 1.5000
  NaN remaining: 0

Saved: tcn_predictions_2020.npy (8760 hours)
